In [ ]:
# Cell 1: Environment Setup & Package Installation

!pip install -q \
    "langchain==0.3.27" \
    "langchain-community==0.3.27" \
    "langchain-groq==0.3.8" \
    "langchain-huggingface==0.3.1" \
    chromadb \
    pypdf \
    "unstructured[pdf]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 21.2 MB/s eta 0:00:00
  

In [ ]:
# Cell 2: Secure API Key Configuration

import os
import getpass

# Prompts for API key securely without saving or echoing plain text
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass(
        "Enter your Groq API Key: "
    )

print("Groq API key configured successfully!")

Enter your Groq API Key: ··········
Groq API key configured successfully!


In [ ]:
# Cell 3: Loading Custom Data & Chunking

from langchain_community.document_loaders import DirectoryLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter


# Create target data directory

os.makedirs("my_data", exist_ok=True)


# Load all documents from the directory

loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)

raw_documents = loader.load()


# Split documents into smaller semantic chunks

text_splitter = RecursiveCharacterTextSplitter(

chunk_size=500,

chunk_overlap=50

)

documents = text_splitter.split_documents(raw_documents)


print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")

100%|██████████| 1/1 [00:29<00:00, 29.84s/it]

Loaded 1 raw document(s) and split into 360 chunks.


In [ ]:
# Cell 4: Embedding Model & Vector DB Indexing

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma


# Initialize open-source embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


# Store embeddings into Chroma vector database

vectorstore = Chroma.from_documents(

documents=documents,

embedding=embeddings

)


# Set vectorstore as a retriever

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Cell 5: Model Initialization and Domain System Prompt

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate


# Initialize the SLM

llm = ChatGroq(

#model_name="llama-3.1-8b-instant",
model_name="openai/gpt-oss-20b",

temperature=0

)


# Custom domain system prompt

system_prompt = (

"You are a specialized AI assistant for the user's uploaded domain.\n"

"Answer questions strictly using ONLY the provided context below.\n"

"If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'\n\n"

"Context:\n{context}"

)


prompt = ChatPromptTemplate.from_messages([

("system", system_prompt),

("human", "{input}"),

])

In [ ]:
# Cell 6: Pipeline Assembly

from langchain.chains import create_retrieval_chain

from langchain.chains.combine_documents import create_stuff_documents_chain


# Combine prompt and LLM to process context

combine_docs_chain = create_stuff_documents_chain(llm, prompt)


# Assemble full retrieval-augmented generation chain

rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [ ]:
# Cell 7: Testing Your Custom Domain Chatbot


# Test Case 1: In-Domain Query

user_query = "what's inside of a Go-bags?."

response = rag_chain.invoke({"input": user_query})


print("--- DOMAIN QUERY ANSWER ---")

print(response["answer"])


print("\n--- RETRIEVED SOURCE CHUNKS ---")

for i, doc in enumerate(response["context"]):

    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
I cannot answer based on the provided domain data.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/PCDRMMO Manual Of Operations v2.pdf
Chunk 2 Source: my_data/PCDRMMO Manual Of Operations v2.pdf
Chunk 3 Source: my_data/PCDRMMO Manual Of Operations v2.pdf


## Web API integration

The cells below expose the existing `rag_chain` through a temporary HTTPS API for the React website. Run all original cells first and ensure documents were loaded into `my_data/`. The tunnel exists only while this Colab runtime is active.


In [ ]:
# Cell 8: Install API dependencies

!pip install -q fastapi uvicorn pyngrok nest-asyncio


In [ ]:
# Cell 9: Create the RAG API

from typing import Any

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

if "rag_chain" not in globals():
    raise RuntimeError("Run Cells 1–6 first so rag_chain is available.")

app = FastAPI(title="Pasig DRRMO RAG API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "https://cruelart11.github.io",
        "http://localhost:5173",
        "http://127.0.0.1:5173",
    ],
    allow_credentials=False,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Content-Type"],
)

class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=2000)
    history: list[dict[str, Any]] = Field(default_factory=list)

@app.get("/health")
def health():
    return {"status": "online", "rag": "ready"}

@app.post("/chat")
def chat(request: ChatRequest):
    try:
        result = rag_chain.invoke({"input": request.message.strip()})
        sources = []

        for document in result.get("context", []):
            metadata = document.metadata or {}
            sources.append({
                "source": str(metadata.get("source", "Unknown")),
                "page": metadata.get("page"),
            })

        return {
            "answer": result.get("answer", "I cannot answer based on the provided domain data."),
            "sources": sources,
            "mode": "rag",
        }
    except Exception:
        raise HTTPException(
            status_code=500,
            detail="The RAG pipeline could not answer this request.",
        )


In [ ]:
# Cell 10: Start a temporary ngrok tunnel

import getpass
import threading
import time

import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

if "_rag_api_server" in globals():
    _rag_api_server.should_exit = True
    time.sleep(1)

ngrok.kill()
ngrok_token = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(ngrok_token)

_rag_api_config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info",
)
_rag_api_server = uvicorn.Server(_rag_api_config)
_rag_api_thread = threading.Thread(
    target=_rag_api_server.run,
    daemon=True,
)
_rag_api_thread.start()
time.sleep(2)

_rag_api_tunnel = ngrok.connect(8000, "http")
RAG_API_URL = _rag_api_tunnel.public_url

print("RAG API is online:", RAG_API_URL)
print("Health check:", f"{RAG_API_URL}/health")
print("Set the GitHub Actions variable VITE_RAG_API_URL to this base URL.")


In [ ]:
# Cell 11: Test the public endpoint

import requests

health_response = requests.get(f"{RAG_API_URL}/health", timeout=15)
chat_response = requests.post(
    f"{RAG_API_URL}/chat",
    json={"message": "What should be inside a Go-Bag?", "history": []},
    timeout=90,
)

print("Health:", health_response.status_code, health_response.json())
print("Chat:", chat_response.status_code, chat_response.json())
